# 📐 Modelo TCO Sostenible (Escenario MITMA 2026 - 44 Toneladas)

Este análisis desglosa la estructura de costes de explotación para un vehículo articulado de gran tonelaje, cumpliendo con la metodología del **Observatorio de Costes del Ministerio de Transportes**.

La tarifa técnica se calcula mediante la suma de costes fijos (amortizados por km) y costes variables directos:

$$ \text{Tarifa Técnica} (€/km) = \text{Coste Fijo Km} + \text{Coste Variable Km} $$

### 1. Coste Fijo por Kilómetro
Calculado sobre la base anual de gastos que no dependen del movimiento del vehículo:
$$ \text{Coste Fijo Km} = \frac{\sum (\text{Personal} + \text{Amortización} + \text{Seguros} + \text{Indirectos})}{\text{Kilometraje Anual}} $$

### 2. Coste Variable por Kilómetro
Costes directos de la operación por cada unidad de distancia:
$$ \text{Coste Variable Km} = \text{Combustible} + \text{Neumáticos} + \text{Mantenimiento} + \text{AdBlue} $$

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

# =====================================================================
# LÍNEA BASE SOSTENIBLE (REFERENCIAS OFICIALES 2026)
# =====================================================================

fixed_costs_base = {
    "personal_y_dietas": 62_500.0,   # Coste Total Cargado (Salario + Seg. Soc. + Dietas)
    "amortizacion_vehiculo": 19_500.0, # Amortización acelerada por mayor desgaste 44t
    "seguros_y_visados": 4_500.0,    # Seguro RC + Mercancía
    "costes_indirectos": 14_500.0,   # Gestión estructural avanzada
    "fiscalidad_y_otros": 2_000.0     # IVTM y tasas técnicas
}

variable_costs_base = {
    "combustible_diesel": 0.460,    # Consumo real 44t (38L/100km) neto profesional
    "mantenimiento_y_tires": 0.165, # Correctivo + Neumáticos 44t (mayor desgaste)
    "adblue_y_aditivos": 0.007, # 38L/100km diésel -> 1.9L/100km AdBlue @ 0.35€/L
}

KM_ANUALES_REF = 110_000

from logistic_core.utils.cost_estimator import CostEstimator
estimator = CostEstimator(
    fixed_costs_annual=fixed_costs_base,
    variable_costs_km=variable_costs_base,
    annual_km_per_truck=KM_ANUALES_REF
)
print(f"Tarifa Técnica Sostenible (MITMA 2026): {estimator.price_per_km:.4f} €/km")

Tarifa Técnica Sostenible (MITMA 2026): 1.5684 €/km


## 🎯 Escenario Estratégico Objetivo (1.50 €/km)

Buscamos la combinación **mínimamente desviada** de la realidad del mercado (Línea Base MITMA) que justifique una tarifa técnica de **1.5 €/km**.

Utilizaremos un optimizador para ajustar levemente las variables operativas (Km, Diésel, Personal, Amortización e Indirectos).

In [2]:
import numpy as np
from scipy.optimize import minimize

TARGET_RATE = 1.50

def objective(x):
    # x[0]=km, x[1]=fuel, x[2]=salario, x[3]=amort, x[4]=ind
    total_fixed = x[2] + x[3] + x[4] + 4_500.0 + 2_000.0
    total_variable = x[1] + 0.165 + 0.015
    rate = (total_fixed / x[0]) + total_variable
    return (rate - TARGET_RATE)**2

# RANGOS DE TOLERANCIA ESTRICTOS (Justificables)
bounds = [
    (90_000, 120_000),  # Kilometraje
    (0.400, 0.550),   # Combustible
    (58_000, 68_000),   # Salario
    (17_000, 23_000),   # Amortización
    (12_500, 16_500)    # Costes Indirectos
]

# Ejecutamos con método SLSQP y alta precisión para clavar el 1.50
res = minimize(objective, [105_000, 0.45, 62_500, 19_500, 14_500], bounds=bounds, method='SLSQP', tol=1e-12)

if res.success:
    o_km, o_f, o_s, o_a, o_i = res.x
    print("=== ESCENARIO OPTIMIZADO PARA 1.5 €/km ===")
    print(f"Tarifa Objetivo:         {TARGET_RATE:.2f} €/km")
    print(f"Kilometraje Anual:       {o_km:,.0f} km")
    print(f"Combustible (€/km):      {o_f:.3f}")
    print(f"Coste Personal Anual:    {o_s:,.0f} €")
    print(f"Amortización Vehículo:   {o_a:,.0f} €")
    print(f"Costes Indirectos:       {o_i:,.0f} €")
    
    # Verificación Matemática
    t_fix = o_s + o_a + o_i + 4_500.0 + 2_000.0
    t_var = o_f + 0.165 + 0.015
    rate_check = (t_fix / o_km) + t_var
    print(f"\nTarifa Resultante Final: {rate_check:.2f} €/km")
else:
    print("No se halló solución exacta. Revise los rangos.")

=== ESCENARIO OPTIMIZADO PARA 1.5 €/km ===
Tarifa Objetivo:         1.50 €/km
Kilometraje Anual:       106,607 km
Combustible (€/km):      0.400
Coste Personal Anual:    60,859 €
Amortización Vehículo:   17,859 €
Costes Indirectos:       12,859 €

Tarifa Resultante Final: 1.50 €/km


## 🔋 Escenario: Camión Eléctrico (44 Toneladas)

A continuación, modelamos el **Total Cost of Ownership (TCO)** para un vehículo eléctrico de gran tonelaje, considerando los mayores costes de inversión (CAPEX) pero menores costes operativos (OPEX).

In [3]:
# =====================================================================
# ESCENARIOS DE CAPEX — CAMIÓN ELÉCTRICO 44t (2026)
# =====================================================================
# Precio lista (Volvo FH Electric / Mercedes eActros 600): 380.000-460.000€
# Ayudas MOVES III + PERTE VEC: 40.000-80.000€ según comunidad autónoma

CAPEX_SCENARIOS = {
    "optimista":   {"capex_neto": 300_000, "descripcion": "Precio lista 380k€ - ayudas máximas 80k€"},
    "central":     {"capex_neto": 370_000, "descripcion": "Precio lista 420k€ - ayudas medias 50k€"},
    "conservador": {"capex_neto": 450_000, "descripcion": "Precio lista 460k€ - sin ayudas"},
}

VIDA_UTIL_ELECTRICO = 7  # años (batería como factor limitante)

# Escenario seleccionado para el modelo base
CAPEX_SELECCIONADO = "central"
capex_neto = CAPEX_SCENARIOS[CAPEX_SELECCIONADO]["capex_neto"]
amortizacion_anual = capex_neto / VIDA_UTIL_ELECTRICO

print(f"Escenario CAPEX: {CAPEX_SELECCIONADO.upper()}")
print(f"CAPEX Neto: {capex_neto:,.0f} €")
print(f"Amortización Anual: {amortizacion_anual:,.0f} €/año")


Escenario CAPEX: CENTRAL
CAPEX Neto: 370,000 €
Amortización Anual: 52,857 €/año


In [4]:
# =====================================================================
# PARÁMETROS ENERGÉTICOS — SENSIBILIDAD
# =====================================================================
# Fuente consumo: Volvo FH Electric spec sheet (condiciones ideales)
# Ajuste operativo: +20-30% por orografía española y carga real

CONSUMO_KWH_KM = {
    "ideal":      1.10,  # Volvo FH Electric, ruta plana, carga optimizada
    "operativo":  1.35,  # Ajuste +23% por condiciones reales España
    "pesimista":  1.80,  # Orografía adversa, carga máxima constante
}

PRECIO_KWH = {
    "nocturno_propio":   0.08,   # Tarifa valle con instalación propia
    "industrial_medio":  0.22,   # Tarifa industrial referencia Repsol 2026
    "red_publica":       0.45,   # Recarga en puntos públicos ultra-rápidos
}

# Escenario seleccionado para el modelo base
consumo_sel = "operativo"
precio_sel  = "industrial_medio"

coste_energia_km = CONSUMO_KWH_KM[consumo_sel] * PRECIO_KWH[precio_sel]
print(f"Consumo: {CONSUMO_KWH_KM[consumo_sel]} kWh/km | Precio: {PRECIO_KWH[precio_sel]} €/kWh")
print(f"Coste energía: {coste_energia_km:.3f} €/km")


Consumo: 1.35 kWh/km | Precio: 0.22 €/kWh
Coste energía: 0.297 €/km


In [5]:
# =====================================================================
# DATOS ESCENARIO ELÉCTRICO (REFERENCIAS SOTA 2026)
# =====================================================================

fixed_costs_electric = {
    "personal_y_dietas": 62_500.0,    # Estructura idéntica al diésel
    "amortizacion_vehiculo": amortizacion_anual, # Integrado desde el escenario de CAPEX
    "seguros_y_visados": 5_500.0,      # +22% vs diésel. Rango real 2026: 5.000-7.500€
                                      # Riesgo al alza: escasa estadística siniestralidad 
                                      # en pesados eléctricos + prima incendio baterías Li-ion
    "infraestructura_carga": 4_500.0,  # Amortización cargador 150kW + instalación eléctrica
                                      # Rango real: 3.000-8.000€/año según instalación compartida
                                      # Valor central conservador para flota multi-camión
    "costes_indirectos": 14_500.0,
    "fiscalidad_y_otros": 2_000.0
}

variable_costs_electric = {
    "energia_electrica": coste_energia_km,       # Integrado desde la parametrización de energía
    "mantenimiento_y_tires": 0.110,  # Reducción operativa del 33% por simplicidad mecánica
    "adblue_y_aditivos": 0.0          # Emisiones cero
}

from logistic_core.utils.cost_estimator import CostEstimator
est_elect = CostEstimator(
    fixed_costs_annual=fixed_costs_electric,
    variable_costs_km=variable_costs_electric,
    annual_km_per_truck=110_000
)
print(f"Tarifa Técnica Eléctrico (Línea Base Escenario Central): {est_elect.price_per_km:.4f} €/km")


Tarifa Técnica Eléctrico (Línea Base Escenario Central): 1.6966 €/km


### 🔍 Análisis de Sensibilidad — Tarifa Eléctrico (Matriz Operativa)

Sustituyendo el enfoque de retrofitting, realizamos un cálculo iterativo paramétrico combinando consumos reales operativos y costes variables de energía para situar nuestro modelo en la matriz del mercado.

In [6]:
# =====================================================================
# ANÁLISIS DE SENSIBILIDAD — TARIFA ELÉCTRICO
# Variamos consumo y precio de energía para mostrar rango real
# =====================================================================

import pandas as pd
import itertools

consumos = [1.10, 1.35, 1.80]       # kWh/km: ideal, operativo, pesimista
precios  = [0.08, 0.22, 0.45]       # €/kWh: nocturno, industrial, público
KM_ANUALES_REF = 110_000

resultados = []
for c, p in itertools.product(consumos, precios):
    coste_energia = c * p
    fixed_total = sum(fixed_costs_electric.values())  # ya con infra carga corregida
    variable_total = coste_energia + 0.110  # mantenimiento sin AdBlue
    tarifa = (fixed_total / KM_ANUALES_REF) + variable_total
    resultados.append({
        "Consumo (kWh/km)": c,
        "Precio (€/kWh)": p,
        "Energía (€/km)": round(coste_energia, 3),
        "Tarifa (€/km)": round(tarifa, 4)
    })

df_sens = pd.DataFrame(resultados)
print("=== ANÁLISIS DE SENSIBILIDAD — TARIFA ELÉCTRICO ===")
print(df_sens.to_string(index=False))
print(f"\nReferencia diésel (ajustada): 1.5684 €/km")
print(f"Rango eléctrico integral:   {df_sens['Tarifa (€/km)'].min():.4f} — {df_sens['Tarifa (€/km)'].max():.4f} €/km")


=== ANÁLISIS DE SENSIBILIDAD — TARIFA ELÉCTRICO ===
 Consumo (kWh/km)  Precio (€/kWh)  Energía (€/km)  Tarifa (€/km)
             1.10            0.08           0.088         1.4876
             1.10            0.22           0.242         1.6416
             1.10            0.45           0.495         1.8946
             1.35            0.08           0.108         1.5076
             1.35            0.22           0.297         1.6966
             1.35            0.45           0.608         2.0071
             1.80            0.08           0.144         1.5436
             1.80            0.22           0.396         1.7956
             1.80            0.45           0.810         2.2096

Referencia diésel (ajustada): 1.5684 €/km
Rango eléctrico integral:   1.4876 — 2.2096 €/km


### ⚖️ Matriz Comparativa Final: Diésel vs Eléctrico (Escenarios Centrales)
Resumen consolidado para incorporar a la presentación académica del TFM.

In [7]:
# =====================================================================
# TABLA COMPARATIVA FINAL — DIÉSEL vs ELÉCTRICO (Escenario Central)
# =====================================================================

# Re-calcular los fijos y variables del diésel del primer bloque (corregido adblue)
fixed_diesel = 62_500 + 19_500 + 4_500 + 14_500 + 2_000
var_diesel = 0.460 + 0.165 + 0.007
rate_diesel = (fixed_diesel / KM_ANUALES_REF) + var_diesel

# Valores electrico escenario seleccionado
fixed_elec = sum(fixed_costs_electric.values())
var_elec = coste_energia_km + 0.110
rate_elec = (fixed_elec / KM_ANUALES_REF) + var_elec

comparativa = {
    "Partida": [
        "Personal y dietas",
        "Amortización vehículo",
        "Seguros y visados",
        "Infraestructura carga",
        "Costes indirectos",
        "Fiscalidad y otros",
        "TOTAL FIJOS ANUALES",
        "",
        "Energía / Combustible (€/km)",
        "Mantenimiento y neumáticos (€/km)",
        "AdBlue / Aditivos (€/km)",
        "TOTAL VARIABLES (€/km)",
        "",
        "TARIFA TÉCNICA (€/km)",
        "Diferencia vs diésel"
    ],
    "Diésel 44t": [
        "62,500 €",
        "19,500 €",
        "4,500 €",
        "0 €",
        "14,500 €",
        "2,000 €",
        f"{fixed_diesel:,.0f} €",
        "",
        "0.460 €",
        "0.165 €",
        "0.007 €",
        f"{var_diesel:.3f} €",
        "",
        f"{rate_diesel:.4f} €/km",
        "BASE"
    ],
    "Eléctrico 44t": [
        f"{fixed_costs_electric['personal_y_dietas']:,.0f} €",
        f"{fixed_costs_electric['amortizacion_vehiculo']:,.0f} €",
        f"{fixed_costs_electric['seguros_y_visados']:,.0f} €",
        f"{fixed_costs_electric['infraestructura_carga']:,.0f} €",
        f"{fixed_costs_electric['costes_indirectos']:,.0f} €",
        f"{fixed_costs_electric['fiscalidad_y_otros']:,.0f} €",
        f"{fixed_elec:,.0f} €",
        "",
        f"{coste_energia_km:.3f} €",
        f"{variable_costs_electric['mantenimiento_y_tires']:.3f} €",
        f"{variable_costs_electric['adblue_y_aditivos']:.3f} €",
        f"{var_elec:.3f} €",
        "",
        f"{rate_elec:.4f} €/km",
        f"{(rate_elec - rate_diesel):+.4f} €/km ({(rate_elec/rate_diesel - 1)*100:+.2f}%)"
    ]
}

df_comp = pd.DataFrame(comparativa)
from IPython.display import display, Markdown
display(Markdown(df_comp.to_markdown(index=False)))


| Partida                           | Diésel 44t   | Eléctrico 44t         |
|:----------------------------------|:-------------|:----------------------|
| Personal y dietas                 | 62,500 €     | 62,500 €              |
| Amortización vehículo             | 19,500 €     | 52,857 €              |
| Seguros y visados                 | 4,500 €      | 5,500 €               |
| Infraestructura carga             | 0 €          | 4,500 €               |
| Costes indirectos                 | 14,500 €     | 14,500 €              |
| Fiscalidad y otros                | 2,000 €      | 2,000 €               |
| TOTAL FIJOS ANUALES               | 103,000 €    | 141,857 €             |
|                                   |              |                       |
| Energía / Combustible (€/km)      | 0.460 €      | 0.297 €               |
| Mantenimiento y neumáticos (€/km) | 0.165 €      | 0.110 €               |
| AdBlue / Aditivos (€/km)          | 0.007 €      | 0.000 €               |
| TOTAL VARIABLES (€/km)            | 0.632 €      | 0.407 €               |
|                                   |              |                       |
| TARIFA TÉCNICA (€/km)             | 1.5684 €/km  | 1.6966 €/km           |
| Diferencia vs diésel              | BASE         | +0.1282 €/km (+8.18%) |

### 📚 Bibliografía y Fuentes Técnicas

1. **Observatorio de Costes del Transporte (MITMA 2026)**: Estructura de costes para vehículos de 44 toneladas.
2. **Volvo FH Electric - Especificaciones TCO**: Consumos reales (1.1 kWh/km) y planes de mantenimiento preventivo.
3. **El Economista (Artículo: Paridad TCO 2026)**: Análisis de la convergencia de costes entre diésel y eléctrico en flotas industriales.
4. **Repsol Soluciones de Energía**: Tarifas industriales de recarga ultra-rápida (0.22 €/kWh proyectado).